# TAREA 3

- Usar dataframes de PySpark para manipular datos
- Dataset: **Austin Crash Report Data - Crash Level Records** el cual es el mismo que desde la tara 1
  - Registros de accidentes viales reportados en Austin, Texas
  - Contiene información sobre severidad, ubicación, tipo de colisión, víctimas, etc.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AustinCrashesAnalysis") \
    .getOrCreate()

In [ ]:
# Leemos el CSV — este dataset tiene encabezados y tipos mixtos,
# por eso usamos inferSchema=True para que Spark detecte enteros, booleanos, etc.
df = spark.read.csv(
    "/content/2_Austin_Crash_Report_Data_-_Crash_Level_Records.csv",
    header=True,
    inferSchema=True,
    nullValue=""
)

In [ ]:
# Vista rápida del esquema y primeras filas
df.printSchema()

In [ ]:
# Seleccionamos las columnas más relevantes para trabajar con ellas
# (el dataset tiene 48 columnas — trabajar con todas es poco práctico)
columnas_interes = [
    "Crash ID", "crash_fatal_fl", "crash_speed_limit",
    "crash_sev_id", "tot_injry_cnt", "death_cnt",
    "units_involved", "road_constr_zone_fl",
    "pedestrian_death_count", "pedestrian_serious_injury_count",
    "bicycle_death_count", "motor_vehicle_death_count",
    "Address", "Crash timestamp (US/Central)"
]

df = df.select(columnas_interes)
df.show(5, truncate=False)

In [ ]:
from pyspark.sql.functions import col, upper, trim, regexp_replace

#a columna 'Address' tiene formatos inconsistentes (mayúsculas/minúsculas mezcladas).
#La mandamos toda a mayúsculas y quitamos espacios sobrantes al inicio y fin,
df = df.withColumn("Address", upper(trim(col("Address"))))

#crash_speed_limit puede venir como string en algunos casos — forzamos a entero
df = df.withColumn("crash_speed_limit", col("crash_speed_limit").cast("integer"))

df.select("Crash ID", "Address", "crash_speed_limit").show(8, truncate=False)


## Agregar nuevas columnas calculadas

In [ ]:
from pyspark.sql.functions import (
    when, to_timestamp, to_date, hour, lit
)

# --- Columna 1: etiqueta de severidad ---
# crash_sev_id es un código numérico que representa la gravedad del accidente:
# 1 = Fatal, 2 = Lesión grave, 3 = Lesión leve, 4 = Posible lesión, 5 = Sin lesión
df = df.withColumn(
    "severidad",
    when(col("crash_sev_id") == 1, "Fatal")
    .when(col("crash_sev_id") == 2, "Lesion grave")
    .when(col("crash_sev_id") == 3, "Lesion leve")
    .when(col("crash_sev_id") == 4, "Posible lesion")
    .otherwise("Sin lesion")
)

In [ ]:
# Si la suma de muertes/heridos graves entre peatones y ciclistas es mayor a 0,
# clasificamos el accidente como uno que involucra usuarios vulnerables
df = df.withColumn(
    "victima_vulnerable",
    (
        col("pedestrian_death_count") + col("pedestrian_serious_injury_count") +
        col("bicycle_death_count")
    ) > 0
)

In [ ]:
# Parseamos el timestamp al formato correcto y luego extraemos
# la fecha limpia y la hora del incidente
df = df.withColumn(
    "ts", to_timestamp(col("`Crash timestamp (US/Central)`"), "yyyy/MM/dd HH:mm:ss")
)
df = df.withColumn("fecha_accidente", to_date(col("ts")))
df = df.withColumn("hora_accidente", hour(col("ts")))

In [ ]:
# --- Columna 5: turno del accidente (mañana / tarde / noche) ---
# Agrupamos por rango de hora para saber en qué parte del día ocurrió el accidente
df = df.withColumn(
    "turno",
    when((col("hora_accidente") >= 6) & (col("hora_accidente") < 12), "Mañana")
    .when((col("hora_accidente") >= 12) & (col("hora_accidente") < 20), "Tarde")
    .otherwise("Noche")
)

# Quitamos la columna auxiliar 'ts' que ya no necesitamos
df = df.drop("ts")

# Vista de las nuevas columnas
df.select(
    "Crash ID", "crash_sev_id", "severidad",
    "victima_vulnerable", "fecha_accidente",
    "hora_accidente", "turno"
).show(15)

## Filtros
Aplicamos distintos filtros para explorar subconjuntos de datos relevantes.

In [ ]:
# Filtro 1: accidentes FATALES ocurridos en zona de construcción
# Equivalente SQL:
# SELECT * FROM crashes WHERE crash_fatal_fl = TRUE AND road_constr_zone_fl = TRUE
df.filter(
    (col("crash_fatal_fl") == True) & (col("road_constr_zone_fl") == True)
).select(
    "Crash ID", "Address", "severidad", "death_cnt",
    "crash_speed_limit", "fecha_accidente"
).show(10, truncate=False)

In [ ]:
# Filtro 2: accidentes con víctimas vulnerables (peatones o ciclistas),
# ordenados por número total de heridos de mayor a menor
df.filter(col("victima_vulnerable") == True) \
  .orderBy(col("tot_injry_cnt").desc()) \
  .select(
      "Crash ID", "Address", "pedestrian_death_count",
      "pedestrian_serious_injury_count", "bicycle_death_count",
      "tot_injry_cnt", "turno"
  ).show(15, truncate=False)

In [ ]:
# Filtro 3: accidentes en vías con límite de velocidad mayor a 60 mph
# que además resultaron en al menos una muerte
# Similar a: SELECT * FROM crashes WHERE crash_speed_limit > 60 AND death_cnt >= 1
df_alta_velocidad = df.filter(
    (col("crash_speed_limit") > 60) & (col("death_cnt") >= 1)
)
df_alta_velocidad \
    .select("Crash ID", "Address", "crash_speed_limit", "death_cnt", "severidad", "turno") \
    .orderBy(col("crash_speed_limit").desc()) \
    .show(10, truncate=False)

In [ ]:
# Filtro 4: accidentes que ocurrieron en la autopista IH 35
# (la más transitada de Austin), usando like() igual que un LIKE '%IH 35%' en SQL
# Filtramos además solo los que ocurrieron en turno Noche para ver el patrón nocturno
df.filter(
    (trim(col("Address")).like("%IH 35%")) & (col("turno") == "Noche")
).select(
    "Crash ID", "Address", "severidad",
    "crash_speed_limit", "tot_injry_cnt",
    "hora_accidente", "fecha_accidente"
).show(10, truncate=False)